# LegalQA Task 2 — Canonical Reproducible Dual-T4 Kaggle Pipeline (Stack A)
Production LegalQA training, validation, and inference pipeline on Kaggle Dual NVIDIA T4 GPUs.
- **Stack A Production Architecture**: Exact/Similar QA Memory -> BM25S (mmap) + Dense DEk21 v2 (FP16 GPU top-K) -> RRF Fusion -> Task-Tuned BGE Reranker -> Structured Evidence Packer -> Qwen2.5-3B-Instruct (4-bit QLoRA) -> Candidate Ensemble & Selector Guardrail.
- **Hardware Layout**: Dual NVIDIA T4 (GPU 0: Qwen Generator | GPU 1: DEk21 Dense + BGE Reranker | CPU: BM25S + QA Memory + Selector).

In [ ]:
# Cell 1 — Execution Profile & Authoritative Production Selection Configuration
import os, sys

SEED = 42
ALLOW_SINGLE_GPU_SMOKE = False  # Strict dual-T4 by default; set True only for explicit single-GPU smoke testing
ALLOW_UNVALIDATED_FINAL = False  # Fail-safe: final profile refuses UNVALIDATED config unless explicitly overridden

# Execution Profiles:
#   'smoke_only': Fast code & hardware verification (~30 optimizer steps, 5 test queries)
#   'screen_fold0': Excludes fold 0 from training, runs S0/S1/S2 bake-off, and generates promotion_report.json
#   'final_train_and_submit': Trains only promoted components on ALL allowed data and creates public submission
#   'reuse_final_checkpoints_and_submit': Reuses verified final all-data checkpoints and creates public submission
EXECUTION_PROFILE = "smoke_only"  # P0-6: Safe committed default

# Smoke bounded limits (P0-7)
SMOKE_MAX_STEPS = 30
SMOKE_MAX_RERANKER_PAIRS = 256
SMOKE_MAX_RERANKER_VAL_PAIRS = 128
SMOKE_MAX_QA_EXAMPLES = 128
SMOKE_EVAL_QUERIES = 5

# Staging path configuration
PRODUCTION_CONFIG_PATH = "configs/production_selection.yaml"

print("===========================================================")
print(f"COMMITTED EXECUTION PROFILE: {EXECUTION_PROFILE}")
print(f"RANDOM SEED:                 {SEED}")
print(f"PRODUCTION CONFIG PATH:      {PRODUCTION_CONFIG_PATH}")
print("===========================================================")

In [ ]:
# Cell 2 — Environment, Secrets & Hardware Verification
import os, sys, gc, glob, json, zipfile, re, math, time, subprocess
from collections import Counter, defaultdict
import numpy as np
import pandas as pd
import torch
from tqdm.auto import tqdm

# Set deterministic seeds
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

# Securely retrieve HuggingFace Token from Kaggle Secrets
try:
    from kaggle_secrets import UserSecretsClient
    user_secrets = UserSecretsClient()
    HF_TOKEN = user_secrets.get_secret("HF_TOKEN")
    if HF_TOKEN:
        os.environ["HF_TOKEN"] = HF_TOKEN
        os.environ["HUGGING_FACE_HUB_TOKEN"] = HF_TOKEN
        print("HF_TOKEN securely loaded from Kaggle Secrets.")
except Exception:
    HF_TOKEN = os.environ.get("HF_TOKEN")
    if HF_TOKEN:
        print("HF_TOKEN detected from environment.")
    else:
        print("Notice: HF_TOKEN secret not present; using public weights.")

# Dual-GPU Device Allocation & Preflight Gate (P0-11)
gpu_count = torch.cuda.device_count() if torch.cuda.is_available() else 0
print(f"CUDA GPUs Detected: {gpu_count}")

if torch.cuda.is_available():
    if gpu_count < 2 and not ALLOW_SINGLE_GPU_SMOKE:
        raise RuntimeError(
            f"Production execution requires 2 CUDA GPUs (Dual-T4), but found {gpu_count}. "
            f"Set ALLOW_SINGLE_GPU_SMOKE=True only for local or single-GPU testing."
        )

if gpu_count >= 2:
    GEN_DEVICE = "cuda:0"
    RETRIEVAL_DEVICE = "cuda:1"
elif gpu_count == 1:
    GEN_DEVICE = "cuda:0"
    RETRIEVAL_DEVICE = "cuda:0"
else:
    GEN_DEVICE = "cpu"
    RETRIEVAL_DEVICE = "cpu"

for i in range(gpu_count):
    p = torch.cuda.get_device_properties(i)
    print(f"  GPU {i}: {p.name} | VRAM: {p.total_memory / (1024**3):.1f} GB | Compute: sm_{p.major}{p.minor}")
print(f"Hardware Allocation -> Generator: {GEN_DEVICE} | Retrieval/Reranker: {RETRIEVAL_DEVICE}")

In [ ]:
# Cell 3 — Deterministic Path Resolution for Runtime Code, Models & Data (P0-14)
# 1. Resolve Code Root containing src/ and scripts/
code_candidates = glob.glob("/kaggle/input/**/code/LegalQA", recursive=True) + [".", "/kaggle/working", "/kaggle/working/LegalQA"]
resolved_code_root = None
for cand in code_candidates:
    if os.path.exists(os.path.join(cand, "src")) and os.path.exists(os.path.join(cand, "scripts")):
        resolved_code_root = os.path.abspath(cand)
        if resolved_code_root not in sys.path:
            sys.path.insert(0, resolved_code_root)
        print(f"Packaged Code Root resolved: {resolved_code_root}")
        break

assert resolved_code_root is not None, "Failed to resolve packaged code directory containing src/ and scripts/!"

from src.task2.path_resolver import find_runtime_roots, find_qwen_model_dir, resolve_runtime_paths
from src.task2.production_config import load_production_selection, validate_production_selection_for_profile

# 2. Resolve production configuration
if os.path.exists(PRODUCTION_CONFIG_PATH):
    resolved_prod_cfg_path = PRODUCTION_CONFIG_PATH
else:
    resolved_prod_cfg_path = os.path.join(resolved_code_root, "configs", "production_selection.yaml")

PRODUCTION_CFG = load_production_selection(resolved_prod_cfg_path)
validate_production_selection_for_profile(
    PRODUCTION_CFG,
    EXECUTION_PROFILE,
    allow_unvalidated_final=ALLOW_UNVALIDATED_FINAL,
)

REQUIRES_GENERATOR = PRODUCTION_CFG.requires_generator

# Derive profile-specific training & evaluation switches
if EXECUTION_PROFILE == "smoke_only":
    RUN_RERANKER_TRAINING = True
    RUN_GENERATOR_TRAINING = True
    RUN_DEV_EVALUATION = True
    RUN_PUBLIC_INFERENCE = False
    REUSE_EXISTING_CHECKPOINTS = False
    TRAIN_VAL_FOLD = 0
    MAX_RERANKER_STEPS = SMOKE_MAX_STEPS
    MAX_RERANKER_PAIRS = SMOKE_MAX_RERANKER_PAIRS
    MAX_RERANKER_VAL_PAIRS = SMOKE_MAX_RERANKER_VAL_PAIRS
    MAX_GENERATOR_STEPS = SMOKE_MAX_STEPS
    MAX_GENERATOR_EXAMPLES = SMOKE_MAX_QA_EXAMPLES
    DEV_EVAL_SIZE = SMOKE_EVAL_QUERIES
elif EXECUTION_PROFILE == "screen_fold0":
    RUN_RERANKER_TRAINING = True
    RUN_GENERATOR_TRAINING = True
    RUN_DEV_EVALUATION = True
    RUN_PUBLIC_INFERENCE = False
    REUSE_EXISTING_CHECKPOINTS = False
    TRAIN_VAL_FOLD = 0
    MAX_RERANKER_STEPS = None
    MAX_RERANKER_PAIRS = None
    MAX_RERANKER_VAL_PAIRS = None
    MAX_GENERATOR_STEPS = None
    MAX_GENERATOR_EXAMPLES = None
    DEV_EVAL_SIZE = 250
elif EXECUTION_PROFILE == "final_train_and_submit":
    RUN_RERANKER_TRAINING = PRODUCTION_CFG.use_task_tuned_reranker
    RUN_GENERATOR_TRAINING = (PRODUCTION_CFG.use_qlora and REQUIRES_GENERATOR)
    RUN_DEV_EVALUATION = False
    RUN_PUBLIC_INFERENCE = True
    REUSE_EXISTING_CHECKPOINTS = False
    TRAIN_VAL_FOLD = None  # All allowed data
    MAX_RERANKER_STEPS = None
    MAX_RERANKER_PAIRS = None
    MAX_RERANKER_VAL_PAIRS = None
    MAX_GENERATOR_STEPS = None
    MAX_GENERATOR_EXAMPLES = None
    DEV_EVAL_SIZE = None
elif EXECUTION_PROFILE == "reuse_final_checkpoints_and_submit":
    RUN_RERANKER_TRAINING = False
    RUN_GENERATOR_TRAINING = False
    RUN_DEV_EVALUATION = False
    RUN_PUBLIC_INFERENCE = True
    REUSE_EXISTING_CHECKPOINTS = True
    TRAIN_VAL_FOLD = None
    MAX_RERANKER_STEPS = None
    MAX_RERANKER_PAIRS = None
    MAX_RERANKER_VAL_PAIRS = None
    MAX_GENERATOR_STEPS = None
    MAX_GENERATOR_EXAMPLES = None
    DEV_EVAL_SIZE = None

# 3. Deterministic Runtime Paths
paths = resolve_runtime_paths("/kaggle/input")
DATA_DIR = paths["data_dir"]
BM25_DIR = paths["bm25_dir"]
DEK21_DIR = paths["dek21_dir"]
MODEL_PATH = paths["qwen_model_path"]

QA_PATH = os.path.join(DATA_DIR, "qa_unique.parquet")
CHUNKS_PATH = os.path.join(DATA_DIR, "legal_chunks.parquet")
KNOWN_QA_PATH = os.path.join(DATA_DIR, "known_qa.json")
TEST_PATH = os.path.join(DATA_DIR, "public-official.json")
if not os.path.exists(TEST_PATH):
    test_files = glob.glob("/kaggle/input/**/public-official.json", recursive=True) or glob.glob("artifacts/**/public-official.json", recursive=True)
    TEST_PATH = test_files[0] if test_files else TEST_PATH

print("Resolved Paths:")
print(f" - Data Directory:      {DATA_DIR}")
print(f" - BM25 Index:          {BM25_DIR}")
print(f" - Dense Index:         {DEK21_DIR}")
print(f" - Qwen Model Path:     {MODEL_PATH}")
print(f" - Production Status:   {PRODUCTION_CFG.status}")
print(f" - Train Reranker:      {RUN_RERANKER_TRAINING}")
print(f" - Train QLoRA:         {RUN_GENERATOR_TRAINING} (Requires Generator: {REQUIRES_GENERATOR})")

In [ ]:
# Cell 4 — Dependency Compatibility & Environment Manifest (Sec 18)
req_path = os.path.join(resolved_code_root, "requirements-kaggle.txt")
if os.path.exists(req_path):
    print(f"Validating dependencies from {req_path}...")
    try:
        subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "-r", req_path])
        print("Dependencies successfully verified!")
    except Exception as e:
        print(f"Warning during pip install: {e}", file=sys.stderr)

import importlib.metadata as md
pkg_versions = {}
for pkg in ["torch", "transformers", "peft", "trl", "bitsandbytes", "sentence-transformers", "bm25s", "nltk", "pyvi", "pandas", "numpy", "scikit-learn"]:
    try:
        ver = md.version(pkg)
        pkg_versions[pkg] = ver
        print(f" - {pkg:25s}: {ver}")
    except Exception:
        pkg_versions[pkg] = "not installed"
        print(f" - {pkg:25s}: not installed")

# Save environment manifest for reproducibility audit
os.makedirs("/kaggle/working", exist_ok=True)
env_manifest = {
    "python": sys.version,
    "packages": pkg_versions,
    "cuda_available": torch.cuda.is_available(),
    "gpu_count": torch.cuda.device_count() if torch.cuda.is_available() else 0,
    "timestamp": time.strftime("%Y-%m-%d %H:%M:%S UTC", time.gmtime()),
}
with open("/kaggle/working/kaggle_environment.json", "w", encoding="utf-8") as f:
    json.dump(env_manifest, f, indent=2)
print("Saved /kaggle/working/kaggle_environment.json")

In [ ]:
# Cell 5 — Strict Production Preflight Diagnostics (P0-11, P0-12, P0-13)
from scripts.preflight_kaggle import run_preflight_checks
from src.common.dense import DenseRetriever

preflight_res = run_preflight_checks(
    pipeline_config_path="configs/pipeline.yaml" if os.path.exists("configs/pipeline.yaml") else os.path.join(resolved_code_root, "configs/pipeline.yaml"),
    models_config_path="configs/models.yaml" if os.path.exists("configs/models.yaml") else os.path.join(resolved_code_root, "configs/models.yaml"),
    production_config_path=resolved_prod_cfg_path,
    require_cuda=torch.cuda.is_available(),
    expected_gpu_count=2,
    allow_single_gpu=ALLOW_SINGLE_GPU_SMOKE,
    check_dataset_files=True,
    data_dir=DATA_DIR,
    bm25_dir=BM25_DIR,
    dek21_dir=DEK21_DIR,
    public_path=TEST_PATH,
    stack="stack_a",
    require_training_files=RUN_RERANKER_TRAINING,
    verify_dense_hash=(EXECUTION_PROFILE == "final_train_and_submit"),
)

if not preflight_res["passed"]:
    print("Preflight validation FAILED:", preflight_res["errors"])
    raise RuntimeError(f"PREFLIGHT FAILED: {preflight_res['errors']}")

# P0-12: Strict preliminary load of Dense index to verify FP16 dtype and shape before long training
print("Executing pre-training strict Dense index probe...")
probe_dense = DenseRetriever.load_index(
    DEK21_DIR,
    corpus_path=CHUNKS_PATH,
    device=RETRIEVAL_DEVICE,
    expected_model_name="CODE4LIFEOFFICIAL/huydang-dek21-embedding-v2",
    expected_dtype="float16",
    final_mode=True,
)
print(f"Dense DEk21 probe successful: {probe_dense.corpus_embeddings.shape} on {RETRIEVAL_DEVICE}")
del probe_dense
if torch.cuda.is_available():
    torch.cuda.empty_cache()

print("Production preflight and index integrity completely PASSED!")

In [ ]:
# Cell 6 — Load QA & Index Metadata without Duplication
from src.task2.qa_memory import QAMemory
from src.common.bm25 import BM25Retriever

print("Loading verified QA Memory...")
memory = QAMemory.load(KNOWN_QA_PATH, QA_PATH)
print(f"Loaded QA Memory: {len(memory.id_to_answer):,} IDs | {len(memory.question_to_answer):,} unique questions.")

print("Loading BM25 Index (mmap)... ")
bm25 = BM25Retriever.load(BM25_DIR, corpus_path=CHUNKS_PATH, fail_on_missing_index=True)
print(f"BM25 Ready: {bm25.corpus_size:,} chunks indexed.")

In [ ]:
# Cell 7 — Task-Tuned Reranker Fine-Tuning & Strict Manifest Validation
from src.task2.checkpoint_manifest import assert_final_checkpoint

RERANKER_CHECKPOINT = "BAAI/bge-reranker-v2-m3"

if RUN_RERANKER_TRAINING:
    from src.task2.training.train_reranker import train_bge_reranker
    pairs_path = os.path.join(DATA_DIR, "reranker_training_pairs.parquet")
    reranker_out = "/kaggle/working/checkpoints/reranker/best"
    print(f"Starting Reranker fine-tuning on {RETRIEVAL_DEVICE} (val_fold={TRAIN_VAL_FOLD}, max_steps={MAX_RERANKER_STEPS}, max_pairs={MAX_RERANKER_PAIRS})...")
    res_rerank = train_bge_reranker(
        pairs_path=pairs_path,
        output_dir=reranker_out,
        model_name="BAAI/bge-reranker-v2-m3",
        epochs=1,
        batch_size=2,
        grad_accum=4,
        lr=2e-5,
        val_fold=TRAIN_VAL_FOLD,
        max_steps=MAX_RERANKER_STEPS,
        max_train_pairs=MAX_RERANKER_PAIRS,
        max_val_pairs=MAX_RERANKER_VAL_PAIRS,
        device=RETRIEVAL_DEVICE,
        fail_on_error=True,
    )
    if res_rerank.get("status") != "completed":
        raise RuntimeError(f"Reranker training requested but failed: {res_rerank}")
    RERANKER_CHECKPOINT = reranker_out
    print(f"Reranker training complete! Best checkpoint saved to {RERANKER_CHECKPOINT}")
elif REUSE_EXISTING_CHECKPOINTS and PRODUCTION_CFG.use_task_tuned_reranker:
    ckpt_candidates = glob.glob("/kaggle/input/**/checkpoints/reranker/best", recursive=True) or glob.glob("checkpoints/reranker/best", recursive=True)
    if not ckpt_candidates:
        raise FileNotFoundError("REUSE_FINAL_CHECKPOINTS profile requested but no pre-trained reranker checkpoint found!")
    RERANKER_CHECKPOINT = ckpt_candidates[0]
    # P0-10: Strict final manifest validation
    assert_final_checkpoint(RERANKER_CHECKPOINT, expected_base_model="BAAI/bge-reranker-v2-m3", component_name="reranker")
    print(f"Reusing verified final reranker checkpoint: {RERANKER_CHECKPOINT}")
else:
    print(f"Using pretrained base reranker: {RERANKER_CHECKPOINT}")

In [ ]:
# Cell 8 — Qwen2.5-3B QLoRA SFT Fine-Tuning & Reload Verification (P0-9, P0-10, Sec 6, 7)
from src.task2.checkpoint_manifest import assert_final_checkpoint

ADAPTER_PATH = None

if RUN_GENERATOR_TRAINING:
    from src.task2.training.train_generator import run_qlora_training
    labels_path = os.path.join(DATA_DIR, "retrieval_labels.parquet")
    qlora_out = "/kaggle/working/checkpoints/generator/hf_adapter"
    print(f"Starting QLoRA fine-tuning on {GEN_DEVICE} (val_fold={TRAIN_VAL_FOLD}, max_steps={MAX_GENERATOR_STEPS}, max_examples={MAX_GENERATOR_EXAMPLES})...")
    res_qlora = run_qlora_training(
        model_name=MODEL_PATH,
        qa_path=QA_PATH,
        labels_path=labels_path,
        chunks_path=CHUNKS_PATH,
        output_dir=qlora_out,
        epochs=1,
        batch_size=1,
        grad_accum=8,
        lr=1e-4,
        max_seq_len=2048,
        val_fold=TRAIN_VAL_FOLD,
        max_steps=MAX_GENERATOR_STEPS,
        max_train_examples=MAX_GENERATOR_EXAMPLES,
        device=GEN_DEVICE,
        fail_on_error=True,
    )
    if res_qlora.get("status") != "completed":
        raise RuntimeError(f"QLoRA training requested but failed: {res_qlora}")
    ADAPTER_PATH = qlora_out
    print(f"QLoRA training complete! Adapter verified and saved to {ADAPTER_PATH}")
elif REUSE_EXISTING_CHECKPOINTS and PRODUCTION_CFG.use_qlora and REQUIRES_GENERATOR:
    ad_candidates = glob.glob("/kaggle/input/**/checkpoints/generator/hf_adapter", recursive=True) or glob.glob("checkpoints/generator/hf_adapter", recursive=True)
    if not ad_candidates:
        raise FileNotFoundError("REUSE_FINAL_CHECKPOINTS profile requested QLoRA but no pre-trained adapter found!")
    ADAPTER_PATH = ad_candidates[0]
    # P0-10: Strict final manifest validation
    assert_final_checkpoint(ADAPTER_PATH, expected_base_model=PRODUCTION_CFG.generator_base_model, component_name="generator")
    print(f"Reusing verified final QLoRA adapter: {ADAPTER_PATH}")
else:
    if not REQUIRES_GENERATOR:
        print(f"QLoRA training/reuse bypassed: Candidate policy '{PRODUCTION_CFG.candidate_policy}' (fixed='{PRODUCTION_CFG.best_fixed_candidate}') does not require generator.")
    else:
        print("QLoRA training skipped by configuration.")

In [ ]:
# Cell 9 — Exact Parameter Audit for Actually Loaded Inference Stack
from scripts.audit_parameters import audit_parameter_budget

models_cfg = "configs/models.yaml" if os.path.exists("configs/models.yaml") else os.path.join(resolved_code_root, "configs/models.yaml")
adapter_manifest = os.path.join(ADAPTER_PATH, "generator_manifest.json") if (ADAPTER_PATH and os.path.exists(os.path.join(ADAPTER_PATH, "generator_manifest.json"))) else (os.path.join(ADAPTER_PATH, "training_manifest.json") if ADAPTER_PATH else None)

audit_res = audit_parameter_budget(models_cfg, stack="stack_a", adapter_manifest_path=adapter_manifest)
print("=== FINAL STACK PARAMETER AUDIT ===")
for k, v in audit_res["breakdown"].items():
    print(f" - {k:45s}: {v:,} parameters")
print(f"TOTAL LEARNED PARAMETERS: {audit_res['total_learned_parameters']:,}")
print(f"OFFICIAL LIMIT:            {audit_res['limit']:,} (strict exclusive)")
print(f"REMAINING SAFE MARGIN:     {audit_res['margin']:,}")
print(f"COMPLIANCE STATUS:         {'COMPLIANT' if audit_res['is_compliant'] else 'NON-COMPLIANT'}")

if not audit_res["is_compliant"]:
    raise RuntimeError(f"PARAMETER BUDGET EXCEEDED: {audit_res['total_learned_parameters']:,} >= {audit_res['limit']:,}")

In [ ]:
# Cell 10 — Real Held-Out Checkpoint Evaluation & Promotion Screen (P0-1, P0-2, P0-5, P0-8, Sec 5, 21)
if RUN_DEV_EVALUATION:
    from src.task2.evaluation import evaluate_checkpoint, run_screen_matrix
    eval_fold = TRAIN_VAL_FOLD if TRAIN_VAL_FOLD is not None else 0
    print(f"Running evaluation on held-out fold {eval_fold} (sample_size={DEV_EVAL_SIZE})...")

    if EXECUTION_PROFILE == "screen_fold0":
        print("Executing Full S0 / S1 / S2 Bake-Off Matrix and generating promotion report...")
        screen_report = run_screen_matrix(
            qa_path=QA_PATH,
            fold_path=os.path.join(DATA_DIR, "fold_assignments.parquet"),
            chunks_path=CHUNKS_PATH,
            held_out_fold=eval_fold,
            bm25_dir=BM25_DIR,
            dense_dir=DEK21_DIR,
            dense_model="CODE4LIFEOFFICIAL/huydang-dek21-embedding-v2",
            base_reranker="BAAI/bge-reranker-v2-m3",
            tuned_reranker=RERANKER_CHECKPOINT,
            base_generator=MODEL_PATH,
            adapter_path=ADAPTER_PATH,
            sample_size=DEV_EVAL_SIZE or 250,
            eval_output_dir="/kaggle/working/evaluations",
            gen_device=GEN_DEVICE,
            retrieval_device=RETRIEVAL_DEVICE,
            seed=SEED,
        )
    else:
        # smoke_only: fast 5 queries verification
        eval_res = evaluate_checkpoint(
            qa_path=QA_PATH,
            fold_path=os.path.join(DATA_DIR, "fold_assignments.parquet"),
            chunks_path=CHUNKS_PATH,
            held_out_fold=eval_fold,
            bm25_dir=BM25_DIR,
            dense_dir=DEK21_DIR,
            dense_model="CODE4LIFEOFFICIAL/huydang-dek21-embedding-v2",
            reranker_checkpoint=RERANKER_CHECKPOINT,
            generator_model=MODEL_PATH if REQUIRES_GENERATOR else None,
            adapter_path=ADAPTER_PATH,
            sample_size=DEV_EVAL_SIZE or 5,
            eval_output_dir="/kaggle/working/evaluations",
            gen_device=GEN_DEVICE,
            retrieval_device=RETRIEVAL_DEVICE,
            fail_on_fallback=True,  # P0-8: Never allow fallback in smoke
            seed=SEED,
        )
        print(f"Smoke Evaluation Complete -> Selected METEOR: {eval_res['selected_meteor']:.4f} | Oracle: {eval_res['oracle_meteor']:.4f}")
else:
    print("Dev evaluation skipped for final submission profile.")

In [ ]:
# Cell 11 — Load Dual-T4 Inference Pipeline based on Frozen Production Selection (Sec 7)
from src.task2.predict import LegalQAPipeline
from src.common.dense import DenseRetriever
from src.common.reranker import BGEReranker
from src.task2.evidence_packer import EvidencePacker
from src.task2.generator import QwenGenerator
from src.task2.selector import CandidateSelector

# Clean memory from training
gc.collect()
if torch.cuda.is_available():
    torch.cuda.empty_cache()

print("Initializing Dual-T4 production pipeline based on authoritative production selection...")
# 1. Dense DEk21 on GPU 1 (mmap FP16)
dense = DenseRetriever.load_index(
    DEK21_DIR,
    corpus_path=CHUNKS_PATH,
    device=RETRIEVAL_DEVICE,
    expected_model_name="CODE4LIFEOFFICIAL/huydang-dek21-embedding-v2",
    expected_dtype="float16",
    final_mode=True,
)

# 2. Reranker on GPU 1
print(f"Loading Reranker from {RERANKER_CHECKPOINT} on {RETRIEVAL_DEVICE}...")
reranker = BGEReranker(model_name=RERANKER_CHECKPOINT, device=RETRIEVAL_DEVICE)

# 3. Evidence Packer on CPU
packer = EvidencePacker(bm25.corpus)

# 4. Generator on GPU 0 — loaded only if candidate policy requires generator
generator: Optional[QwenGenerator] = None
if REQUIRES_GENERATOR:
    print(f"Loading Qwen Generator on {GEN_DEVICE} (require_adapter={PRODUCTION_CFG.use_qlora})...")
    generator = QwenGenerator.load(
        model_path=MODEL_PATH,
        adapter_path=ADAPTER_PATH if PRODUCTION_CFG.use_qlora else None,
        device=GEN_DEVICE,
        runtime="torch" if torch.cuda.is_available() else "fallback",
        fail_on_fallback=True,
        final_mode=True,
        require_adapter=PRODUCTION_CFG.use_qlora,
    )
else:
    print(f"Inference generator bypassed: Production candidate policy is '{PRODUCTION_CFG.candidate_policy}' (fixed='{PRODUCTION_CFG.best_fixed_candidate}').")

# 5. Selector on CPU with Frozen Candidate Policy
selector = CandidateSelector(
    policy=PRODUCTION_CFG.candidate_policy,
    best_fixed_candidate=PRODUCTION_CFG.best_fixed_candidate or "stitched_extract",
)

pipeline = LegalQAPipeline(memory, bm25, dense, reranker, packer, generator, selector)
print("Dual-T4 Production Pipeline loaded successfully!")

In [ ]:
# Cell 12 — Load Public Test Set & Execute True Batched Inference (Sec 8)
with open(TEST_PATH, "r", encoding="utf-8") as f:
    public_test = json.load(f)

print(f"Loaded {len(public_test)} public test questions from {TEST_PATH}.")

items_to_predict = [{"id": str(qid), "question": str(item.get("question", "")).strip()} for qid, item in public_test.items()]

if RUN_PUBLIC_INFERENCE:
    start_time = time.time()
    print(f"Executing true batched inference on {len(items_to_predict)} questions...")
    batch_size_gen = 4 if torch.cuda.is_available() else 1
    submission = pipeline.predict_batch(
        items=items_to_predict,
        max_new_tokens=PRODUCTION_CFG.max_new_tokens,
        retrieval_batch_size=32,
        reranker_batch_size=32,
        generation_batch_size=batch_size_gen,
    )
    elapsed = time.time() - start_time
    print(f"Inference completed in {elapsed:.1f}s ({len(submission)} predictions generated).")
else:
    print("Public inference skipped by profile configuration.")
    submission = {}

In [ ]:
# Cell 13 — Strict Submission Verification & Formatting Gates (Sec 26)
if RUN_PUBLIC_INFERENCE:
    print("=== Strict Submission Verification ===")
    assert len(submission) == 1000, f"Submission count mismatch! Expected 1000, got {len(submission)}"

    test_keys = set(public_test.keys())
    sub_keys = set(submission.keys())
    assert test_keys == sub_keys, f"Submission keys mismatch! Diff: {test_keys ^ sub_keys}"

    forbidden_patterns = [
        "<|im_start|>",
        "<|im_end|>",
        "assistant\n",
        "[DOCUMENT]",
        "[ARTICLE]",
        "[CLAUSE]",
        "NaN",
        "undefined",
        "null",
    ]

    for qid, val in submission.items():
        assert isinstance(val, dict) and "answer" in val, f"Invalid entry format for ID {qid}: {val}"
        ans = val.get("answer", "")
        assert isinstance(ans, str) and len(ans.strip()) > 0, f"Empty answer for ID {qid}!"

        for pat in forbidden_patterns:
            assert pat not in ans, f"Forbidden pattern '{pat}' leaked into submission for query ID {qid}!"

    lengths = [len(v["answer"].split()) for v in submission.values()]
    print(f"Total Queries:        {len(submission):,}")
    print(f"Mean Word Count:      {np.mean(lengths):.1f} words")
    print(f"Median Word Count:    {np.median(lengths):.1f} words")
    print(f"P90 Word Count:       {np.percentile(lengths, 90):.1f} words")
    print(f"Min / Max Length:     {np.min(lengths)} / {np.max(lengths)} words")
    print("All strict submission verification gates completely PASSED!")

In [ ]:
# Cell 14 — Save Submission Archive & Provenance Manifest (Sec 25, 26)
if RUN_PUBLIC_INFERENCE:
    out_dir = "/kaggle/working" if os.path.exists("/kaggle/working") else "artifacts/task2/submissions"
    os.makedirs(out_dir, exist_ok=True)

    out_json = os.path.join(out_dir, "submission.json")
    out_zip = os.path.join(out_dir, "submission.json.zip")
    manifest_path = os.path.join(out_dir, "run_manifest.json")

    with open(out_json, "w", encoding="utf-8") as f:
        json.dump(submission, f, ensure_ascii=False, indent=2)

    with zipfile.ZipFile(out_zip, "w", zipfile.ZIP_DEFLATED) as z:
        z.write(out_json, arcname="submission.json")

    # Section 26: Inspect zip archive immediately
    with zipfile.ZipFile(out_zip, "r") as z:
        zip_names = z.namelist()
        assert zip_names == ["submission.json"], f"ZIP archive namelist mismatch! Expected ['submission.json'], got {zip_names}"

    run_manifest = {
        "timestamp": time.strftime("%Y-%m-%d %H:%M:%S UTC", time.gmtime()),
        "execution_profile": EXECUTION_PROFILE,
        "num_queries": len(submission),
        "stack": PRODUCTION_CFG.stack,
        "production_status": PRODUCTION_CFG.status,
        "production_selection_sha256": PRODUCTION_CFG.source_screen_sha256,
        "reranker_checkpoint": RERANKER_CHECKPOINT,
        "adapter_path": ADAPTER_PATH,
        "candidate_policy": PRODUCTION_CFG.candidate_policy,
        "best_fixed_candidate": PRODUCTION_CFG.best_fixed_candidate,
        "requires_generator": REQUIRES_GENERATOR,
        "mean_word_count": float(np.mean(lengths)),
        "learned_parameters": audit_res["total_learned_parameters"],
        "zip_namelist": zip_names,
    }
    with open(manifest_path, "w", encoding="utf-8") as f:
        json.dump(run_manifest, f, indent=2)

    print(f"Created: {out_json} ({os.path.getsize(out_json)/1024:.1f} KB)")
    print(f"Created: {out_zip} ({os.path.getsize(out_zip)/1024:.1f} KB)")
    print(f"Created: {manifest_path}")
    print("\nSUCCESS: Kaggle Stack A pipeline finished, verified, and ready for submission!")
else:
    print(f"\nExecution profile '{EXECUTION_PROFILE}' finished successfully.")